# Partie qualité de l'air : extraction et nettoyage des mesures Airparif

**Source.** Les couches de mesures du WFS Airparif (projet DIDON) ne gardent que les dernières 24 h :
elles étaient vides lors des tests des 18 et 25 octobre 2026. L'historique vient donc des
[fichiers horaires du LCSQA sur data.gouv.fr](https://www.data.gouv.fr/datasets/5b98b648634f415309d52a50),
qui reprennent les mesures des stations Airparif, **en UTC**, avec leurs indicateurs de validité.
Aucune clé d'API n'est nécessaire.

**Période et pas de temps.** Un an de données **horaires**, du 2025-09-01 au 2026-08-31 (voir `config.yaml`).

Toute la logique est dans `src/` : ce notebook ne fait qu'appeler les fonctions et montrer les résultats.

| Étape | Commande | Sortie |
|---|---|---|
| Extraction (depuis la source) | `python -m src.extract.airparif` | `data/raw/airparif/` |
| Envoi sur SSP Cloud (une fois) | `python -m src.stockage envoyer` | `diffusion/projet-qualite-air/` |
| Récupération depuis SSP Cloud | `python -m src.stockage recuperer` | `data/raw/airparif/` |
| Nettoyage | `python -m src.clean.airparif` | `data/interim/airparif/` |

**Données figées sur SSP Cloud.** Les sources en ligne changent (le format du LCSQA a changé au
1er janvier 2026) : les fichiers bruts extraits sont donc figés dans le dossier `diffusion` du stockage
SSP Cloud, lisible par tous les utilisateurs de la plateforme. Le notebook les récupère de là, puis
refait tout le nettoyage à partir des données brutes.

**Ce que la suite du projet récupère** : `df_air`, une ligne par station et par heure UTC (début de période),
une colonne par polluant en µg/m³, et `df_stations_air`, avec le type de chaque station (trafic / fond)
et ses coordonnées pour la fusion spatiale avec le trafic.

In [ ]:
import os
from pathlib import Path

# Le notebook peut être lancé depuis la racine du projet ou depuis un sous-dossier.
while not Path("config.yaml").exists():
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from src import stockage
from src.clean import airparif as nettoyage_air
from src.extract import airparif as extraction_air
from src.utils import charger_config, configurer_logs

configurer_logs()
config = charger_config()
FIGURES = Path("reports/figures")
config["periode"]

## 1. Les données brutes

Par défaut, les fichiers bruts sont récupérés depuis SSP Cloud (quelques secondes sur la plateforme).
Pour repartir de la source, passer `RELANCER_EXTRACTION` à `True` : le script télécharge alors un
fichier par jour et n'en garde que les lignes Airparif (environ 45 minutes la première fois, avec un cache).

In [ ]:
RELANCER_EXTRACTION = False

if RELANCER_EXTRACTION:
    extraction_air.extraire(config)
elif not (Path(config["chemins"]["raw"]) / "airparif").exists():
    stockage.recuperer(config, dataset="airparif")

## 2. Contrôles juste après l'extraction

Nombre de lignes attendu contre reçu, première et dernière date, colonnes, types.

In [ ]:
brut_air = extraction_air.charger_brut(config)
print("Dimensions :", brut_air.shape)
print("Colonnes :", brut_air.columns.tolist())
brut_air.head()

In [ ]:
debut = pd.to_datetime(brut_air["Date de début"], format="%Y/%m/%d %H:%M:%S", utc=True)
print("Première heure :", debut.min())
print("Dernière heure :", debut.max())

jours = pd.date_range(config["periode"]["debut"], config["periode"]["fin"], freq="D")
print(f"Jours attendus : {len(jours)}, jours reçus : {debut.dt.normalize().nunique()}")

# Une série = une station × un polluant. On attend une ligne par heure.
series = brut_air.groupby(["code site", "Polluant"]).size()
attendu = len(jours) * 24
print(f"Séries : {len(series)}, lignes attendues par série : {attendu}")
print("Séries incomplètes (lignes reçues) :")
series[series != attendu]

In [ ]:
# Toutes les colonnes sont lues en texte : on vérifie ce qu'elles contiennent.
for col in ["Polluant", "type d'influence", "unité de mesure", "code qualité", "validité"]:
    print(col, brut_air[col].value_counts(dropna=False).to_dict())

**Ce que montrent les contrôles** (à reprendre dans le rapport) :

- une ligne par station, polluant et heure : pas de doublon dans la source ;
- `validité` : `1` = valide, `4` = valide (code propre à l'ozone), `-1` = invalide (valeur déjà vide) ;
- `code qualité` : `A` = validé par Airparif, `R` = pas encore validé, `N` = invalide ;
- des petites valeurs négatives, surtout pour le NO : le bruit de l'appareil autour de zéro ;
- le CO est en mg/m³, tout le reste en µg/m³ : le CO n'est pas gardé ;
- depuis le 2026-01-01, la source met chaque champ entre guillemets : le script d'extraction en tient compte.

In [ ]:
# Part des mesures pas encore validées par Airparif (code R), mois par mois.
mois = debut.dt.strftime("%Y-%m")
part_r = (brut_air["code qualité"] == "R").groupby(mois).mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
part_r.mul(100).plot.bar(ax=ax, color="tab:orange")
ax.set_title("Chaque mois, 7 à 13 % des mesures ne sont pas encore validées par Airparif")
ax.set_xlabel("Mois")
ax.set_ylabel("Mesures non validées (%)")
plt.tight_layout()

## 3. Nettoyage

Les règles sont dans `src/clean/airparif.py` et leurs seuils dans `config.yaml`.
Aucun trou n'est bouché ici : une valeur fausse devient `NaN`, l'imputation viendra plus tard.

In [ ]:
tables_air = nettoyage_air.nettoyer(config)
nettoyage_air.sauvegarder(tables_air, config)

### Tableau des décisions de nettoyage

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(tables_air["decisions"])

### Table nettoyée : une ligne par station et par heure (UTC), une colonne par polluant (µg/m³)

In [ ]:
df_air = tables_air["mesures"]
print("Dimensions :", df_air.shape)
print(df_air.dtypes)
df_air.head()

### Les stations
`influence` distingue les stations de bord de route (`trafic`) des stations éloignées des axes (`fond`).

In [ ]:
df_stations_air = tables_air["stations"]
print(df_stations_air["influence"].value_counts())
df_stations_air

## 4. Les trous : couverture des séries, trous courts et pannes longues

In [ ]:
couverture_air = tables_air["couverture"]
print("Séries à faible couverture (signalées, pas supprimées) :")
couverture_air[couverture_air["faible_couverture"]]

In [ ]:
polluants = list(config["airparif"]["polluants"].values())
part_valide = couverture_air.pivot(index="station", columns="polluant", values="part_valide").reindex(columns=polluants)
# Case vide = polluant jamais mesuré par la station (pas un trou).
part_valide.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.0%}", na_rep="")

In [ ]:
trous_air = tables_air["trous"]
print(trous_air.groupby("type")["duree_h"].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 3.5))
trous_air["duree_h"].clip(upper=48).plot.hist(bins=48, ax=ax)
ax.set_title("La plupart des trous ne durent qu'une ou deux heures")
ax.set_xlabel("Durée du trou (heures, regroupées au-delà de 48 h)")
ax.set_ylabel("Nombre de trous")
plt.tight_layout()

## 5. Exemple de capteur défaillant

Une panne de six jours du NO₂ au boulevard Haussmann, fin juin 2026 : la série est simplement
interrompue. Ces heures restent vides dans `df_air` jusqu'à la phase d'imputation.

In [ ]:
station_vue, debut_vue, fin_vue = "FR04131", "2026-06-17", "2026-07-08"
vue = df_air[(df_air["station"] == station_vue) & df_air["heure"].between(debut_vue, fin_vue)]
trous_vue = trous_air[
    (trous_air["station"] == station_vue) & (trous_air["polluant"] == "no2")
    & (trous_air["fin"] >= debut_vue) & (trous_air["debut"] <= fin_vue)
]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(vue["heure"], vue["no2"], lw=0.8)
for _, t in trous_vue.iterrows():
    ax.axvspan(t["debut"], t["fin"], color="tab:red", alpha=0.15 if t["type"] == "long" else 0.05)
ax.set_title("Le capteur de NO₂ du boulevard Haussmann s'est arrêté six jours fin juin 2026")
ax.set_xlabel("Heure (UTC)")
ax.set_ylabel("NO₂ (µg/m³)")
plt.tight_layout()
fig.savefig(FIGURES / "airparif_capteur_defaillant.png", dpi=150)

## 6. Premier regard : le NO₂ d'une station trafic et d'une station de fond

Une semaine d'octobre 2025, pour vérifier que la table a du sens avant l'analyse exploratoire.

In [ ]:
station_trafic, station_fond = "FR04131", "FR04055"  # Boulevard Haussmann, Paris 1er Les Halles
semaine = df_air[df_air["heure"].between("2025-10-06", "2025-10-13", inclusive="left")]
noms = df_stations_air.set_index("station")["nom"]

fig, ax = plt.subplots(figsize=(11, 4))
for code_station in (station_trafic, station_fond):
    s = semaine[semaine["station"] == code_station]
    ax.plot(s["heure"], s["no2"], label=noms[code_station])
ax.set_title("Le NO₂ du boulevard Haussmann dépasse celui des Halles, surtout aux heures de pointe")
ax.set_xlabel("Heure (UTC)")
ax.set_ylabel("NO₂ (µg/m³)")
ax.legend()
plt.tight_layout()

## 7. À retenir pour la suite

- **Clé de fusion** : (`station`, `heure`), heure en UTC, au **début** de la période. Le trafic horodate
  la fin de l'heure : il faudra le décaler d'une heure avant de fusionner.
- **Fusion spatiale** : `latitude` / `longitude` de `df_stations_air` (WGS 84), à projeter en Lambert-93 (EPSG:2154).
- **Stations à surveiller** : RN2-Pantin (FR04123) s'arrête le 2025-10-29 ; les PM₁₀ des Halles (FR04055) s'arrêtent fin 2025.
- **Valeurs négatives mises à NaN** : ces trous ne touchent que les heures très peu polluées (NO surtout),
  un cas MNAR à discuter dans la partie imputation.